In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML

from BadData_AppC import DataObject, AppendixCFunction, Trainer, Overlap

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def run_and_visualize_experiment(config: dict):
    """
    Runs a single experiment based on a configuration dictionary and visualizes the results.
    """
    print("="*80)
    print(f"Starting Experiment: {config['latex_title']} (p={config['p']})")
    print("="*80)

    # 1. Setup
    modular_function = AppendixCFunction(config['c'], config['d'], config['p'])
    dataset = DataObject(modular_function, split=config['split'])
    model = Overlap(config['p'], config['embedding_dim'], config['hidden'])
    model = model.to(DEVICE)
    trainer = Trainer(learning_rate=config['learning_rate'])
    
    num_params = sum(p.numel() for p in model.parameters())
    space_dim = (config['p'] ** 2) * 2**4
    pos_dim = config['p'] ** 2
    print(f"num_parameters: {num_params:,}; space_dim: {space_dim:,}; pos_dim: {pos_dim:,}")
    
    if config.get('print_test_len', False):
        print(len(dataset.test_data))

    # 2. Training
    trainer.train_model(
        model,
        dataset,
        max_steps=config['max_steps'],
        batch_size=config['batch_size'],
        weight_decay=config['weight_decay']
    )

    # Polynomial as a string
    latex_title = config['latex_title']
    p = config['p']

    # 3. Visualization
    # Plot Loss
    plt.figure()
    plt.plot(model.loss_dictionary['train_loss'], label="Train Loss")
    plt.plot(model.loss_dictionary['test_loss'], label="Test Loss", linestyle='--')
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    if len(dataset.test_data) == 1:
        plt.title('Loss ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Loss', fontsize=10)
    plt.legend()

    # Plot Accuracy
    plt.figure()
    plt.plot(model.loss_dictionary['train_accuracy'], label="Train Accuracy")
    plt.plot(model.loss_dictionary['test_accuracy'], label="Test Accuracy")
    plt.xlabel("Training step")
    plt.ylabel("Accuracy")
    if len(dataset.test_data) == 1:
        plt.title('Accuracy ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Accuracy', fontsize=10)
    plt.legend()
    plt.show()

    # Animation for single-point test sets
    if len(dataset.test_data) == 1:

        rcParams['animation.embed_limit'] = 64  # MB, default is 20

        # Histogram
        example = dataset.test_data
        print(example)
        plt.figure()
        plt.bar(list(range(p)), model.loss_dictionary['counts_hist'])
        plt.title("Histogram " + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
        plt.xlabel("Question index")
        plt.ylabel("Count of predictions")

        # Evolution of probability distribution
        probs = []
        for f in model.loss_dictionary.get('prob_dist', []):
            arr = f.detach().cpu().squeeze().numpy() if isinstance(f, torch.Tensor) else np.array(f).squeeze()
            s = float(arr.sum())
            if s > 0:  
                arr = arr / s
            else:
                print("Warning: Sum of probabilities is zero.")
            probs.append(arr)

        if probs:
            fig, ax = plt.subplots(figsize=(6,3))
            ax.set_title('Probability distribution evolution ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)   
            bars = ax.bar(range(len(probs[0])), probs[0])
            ax.set_ylim(0, 1) 
            txt = ax.text(0.02, 0.95, '', transform=ax.transAxes)

            def update(i):
                y = probs[i]
                for b, h in zip(bars, y): 
                    b.set_height(float(h))
                txt.set_text(f"step {i+1}/{len(probs)} | sum={y.sum():.3f} | argmax={int(np.argmax(y))}")
                return bars
            
            anim = FuncAnimation(fig, update, frames=len(probs), interval=100, repeat=False)
            plt.close(fig)
            display(HTML(anim.to_jshtml()))
    
    print("\n✅ Experiment Complete.\n")


In [ ]:
# Polynomial (4*x + y**2)**3 % 97

experiments = [
    {
        "p": 97, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.5,
    },
    # Polynomial (4*x + y**2)**3 + xy % 97
    {
        "p": 97, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (2*x + 3*y)**4 % 97
    {
        "p": 97, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 97
    {
        "p": 97, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 % 97
    {
        "p": 97, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 - y % 97
    {
        "p": 97, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (4*x + y**2)**3 % 23
    {
        "p": 23, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (4*x + y**2)**3 + xy % 23
    {
        "p": 23, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (2*x + 3*y)**4 % 23
    {
        "p": 23, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 23
    {
        "p": 23, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 % 23
    {
        "p": 23, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    #Polynomial (5*x**3 + 2*y**4)**2 - y % 23
    {
        "p": 23, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
]

default_training_params = {
    "max_steps": 1000,
    "learning_rate": 0.005,
    "batch_size": 1024,
    "weight_decay": 1e-3,
}


In [ ]:

for exp_config in experiments:
    final_config = {**default_training_params, **exp_config} # Merge dictionaries
    run_and_visualize_experiment(final_config)